# 基本面多因子策略研究

本 notebook 用于快速浏览数据、因子截面相关性与回测结果。
完整回测请运行 `python scripts/run_backtest.py`。

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.data.data_loader import DataLoader
from src.utils.helpers import load_config

config = load_config()
loader = DataLoader(raw_dir=config['data']['raw_dir'], processed_dir=config['data']['processed_dir'])
print('配置加载完成')

In [ ]:
# 加载估值与财务数据
data = loader.load_data(config['data']['start_date'], config['data']['end_date'])
valuation, financial = data['valuation'], data['financial']
panel = loader.prepare_panel_data(valuation)
print(f"交易日: {panel['ret'].shape[0]}, 股票数: {panel['ret'].shape[1]}")
panel['close'].tail(5)

In [ ]:
# 最近一个截面的估值分布
latest = pd.DataFrame({
    'pe_ttm': panel['pe_ttm'].iloc[-1],
    'pb': panel['pb'].iloc[-1],
    'total_mv': panel['total_mv'].iloc[-1] / 1e8,  # 亿元
}).dropna()
latest.describe()

In [ ]:
# 回测结果汇总
with open(os.path.join(config['results']['report_dir'], 'analysis_results.yaml'), encoding='utf-8') as f:
    results = yaml.safe_load(f)

pd.DataFrame(results['ic_summary']).T

In [ ]:
# 净值曲线与分层结果
from IPython.display import display, Image
plot_dir = config['results']['plot_dir']
for png in ['nav_curve.png', 'quantile_nav.png', 'factor_ic_cumsum.png']:
    display(Image(filename=os.path.join(plot_dir, png)))